In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import pickle
import datetime
from tqdm import tqdm
import scipy.io
import h5py

# 引入自定义的DataLoader
from brain_voxel_dataloader import BrainVoxelDataLoader, BrainVoxelDataset

# 定义与原始代码相同的4×4096全连接网络模型
class Dense4x4096(nn.Module):
    def __init__(self, input_dim=341, num_classes=102):
        super(Dense4x4096, self).__init__()
        self.fc1 = nn.Linear(input_dim, 4096)
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, 4096)
        self.fc4 = nn.Linear(4096, 4096)
        self.fc5 = nn.Linear(4096, num_classes)
        
        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        
        # 添加L2正则化（在优化器中实现）
        self.weight_decay = 0.00001
    
    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.dropout(self.relu(self.fc4(x)))
        x = self.fc5(x)
        return x
    
    def predict(self, x):
        """与Keras兼容的预测方法"""
        self.eval()
        with torch.no_grad():
            logits = self(x)
            probs = self.softmax(logits)
        return probs.cpu().numpy()

def train_model(train_loader, val_loader, input_dim=341, num_classes=102, device='cuda'):
    """训练模型，使用与原代码相同的超参数"""
    # 模型配置
    no_epochs = 25
    learning_rate = 0.00001
    weight_decay = 0.00001
    
    # 创建模型
    model = Dense4x4096(input_dim, num_classes).to(device)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # 训练日志
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    # 最佳模型跟踪
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0
    best_model_state = None
    
    # 创建日志和模型保存目录
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_dir = f'./logs/{timestamp}'
    model_dir = f'./models/{timestamp}'
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    
    # 训练循环
    print("开始训练...")
    for epoch in range(no_epochs):
        # 训练模式
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        # 进度条
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{no_epochs} [Train]')
        
        for inputs, labels in pbar:
            inputs = inputs.to(device)
            if isinstance(labels, tuple):  # 如果标签包含病人ID
                labels = labels[0]
            
            # 将one-hot编码转换为类别索引
            if labels.shape[1] > 1:  # 检查是否为one-hot编码
                labels = torch.argmax(labels, dim=1)
            
            labels = labels.to(device)
            
            # 清零梯度
            optimizer.zero_grad()
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # 反向传播和优化
            loss.backward()
            optimizer.step()
            
            # 统计
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # 更新进度条
            pbar.set_postfix({'loss': train_loss / (pbar.n + 1), 'acc': 100 * correct / total})
        
        # 计算平均训练损失和准确率
        train_loss = train_loss / len(train_loader)
        train_acc = 100 * correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # 验证模式
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{no_epochs} [Val]')
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                if isinstance(labels, tuple):  # 如果标签包含病人ID
                    labels = labels[0]
                
                # 将one-hot编码转换为类别索引
                if labels.shape[1] > 1:  # 检查是否为one-hot编码
                    labels = torch.argmax(labels, dim=1)
                
                labels = labels.to(device)
                
                # 前向传播
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                # 统计
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                # 更新进度条
                pbar.set_postfix({'loss': val_loss / (pbar.n + 1), 'acc': 100 * correct / total})
        
        # 计算平均验证损失和准确率
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        # 打印结果
        print(f'Epoch {epoch+1}/{no_epochs} - '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        # 保存最佳模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(model_dir, 'best_model.pth'))
            print(f'Epoch {epoch+1}: 保存最佳模型，验证损失: {val_loss:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'早停: {patience}个epoch验证损失未改善')
                break
        
        # 保存每个epoch的模型
        torch.save(model.state_dict(), os.path.join(model_dir, f'model_epoch_{epoch+1}.pth'))
    
    # 加载最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f'已加载最佳模型，验证损失: {best_val_loss:.4f}')
    
    # 保存训练日志
    training_log = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
    }
    with open(os.path.join(log_dir, 'training_log.pkl'), 'wb') as f:
        pickle.dump(training_log, f)
    
    # 绘制训练曲线
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Acc')
    plt.plot(val_accs, label='Val Acc')
    plt.title('Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(log_dir, 'training_curves.png'))
    
    return model, training_log

def evaluate_model(model, test_loader, device='cuda'):
    """评估模型在测试集上的性能"""
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc='Evaluating'):
            inputs = inputs.to(device)
            if isinstance(labels, tuple):  # 如果标签包含病人ID
                labels = labels[0]
            
            # 将one-hot编码转换为类别索引
            if labels.shape[1] > 1:  # 检查是否为one-hot编码
                target_labels = torch.argmax(labels, dim=1)
            else:
                target_labels = labels
            
            target_labels = target_labels.to(device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, target_labels)
            
            # 统计
            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += target_labels.size(0)
            correct += (predicted == target_labels).sum().item()
            
            # 保存预测和标签
            all_predictions.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    # 计算平均测试损失和准确率
    test_loss = test_loss / len(test_loader)
    test_acc = 100 * correct / total
    
    print(f'测试集损失: {test_loss:.4f}, 测试集准确率: {test_acc:.2f}%')
    
    # 合并所有批次的预测和标签
    y_pred = np.vstack(all_predictions)
    y_true = np.vstack(all_labels)
    
    return test_loss, test_acc, y_pred, y_true

def apply_model(model, scaler, filepath='DEMO38.mat', save_path=None, device='cuda'):
    """将模型应用到新数据上并保存预测结果"""
    # 加载需要预测的数据
    arrays = {}
    with h5py.File(filepath, 'r') as f:
        for k, v in f.items():
            arrays[k] = np.array(v)
    
    # 转置并标准化数据
    multidim_data = arrays['multidim_data'].transpose()
    multidim_data = scaler.transform(multidim_data)
    
    # 将数据转换为PyTorch张量
    tensor_data = torch.FloatTensor(multidim_data).to(device)
    
    # 使用模型进行预测
    model.eval()
    with torch.no_grad():
        logits = model(tensor_data)
        probs = nn.Softmax(dim=1)(logits)
    
    predicted_ann = probs.cpu().numpy()
    
    # 保存预测结果
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        scipy.io.savemat(
            os.path.join(save_path, 'dense_4x4096_model_prediction.mat'), 
            mdict={'predicted_ann': predicted_ann}
        )
    
    return predicted_ann

def visualize_model_saliency(model, val_dataset, indices=None, device='cuda'):
    """可视化模型显著性图，用于解释模型决策"""
    try:
        import captum
        from captum.attr import Saliency
        import matplotlib.pyplot as plt
        
        # 如果没有指定索引，使用默认值
        if indices is None:
            indices = [10000, 20000, 30000, 40000, 50000, 60000]
            indices = [i for i in indices if i < len(val_dataset)]
        
        # 创建Saliency对象
        saliency = Saliency(model)
        
        # 可视化
        for idx in indices:
            # 获取输入和真实类别
            inputs, labels = val_dataset[idx]
            if isinstance(labels, tuple):  # 如果标签包含病人ID
                labels = labels[0]
            
            # 将数据移到设备
            inputs = inputs.unsqueeze(0).to(device)  # 添加批次维度
            
            # 将one-hot编码转换为类别索引
            if labels.dim() > 0 and labels.size(0) > 1:  # 检查是否为one-hot编码
                target_label = torch.argmax(labels).item()
            else:
                target_label = labels.item()
            
            # 创建目标函数（针对正确类别）
            def target_func(inputs):
                return model(inputs)[:, target_label]
            
            # 计算显著性图
            attributions = saliency.attribute(inputs, target=target_label)
            attributions = attributions.squeeze(0).cpu().numpy()
            
            # 绘制
            plt.figure(figsize=(10, 5))
            plt.title(f"类别: {target_label}")
            plt.plot(attributions, 'g', label='Saliency')
            plt.plot(inputs.squeeze(0).cpu().numpy(), 'b', alpha=0.3, label='Input')
            plt.axvspan(0, 15, color='gray', alpha=0.3)
            plt.axvspan(225, 230, color='gray', alpha=0.2)
            plt.legend()
            plt.grid(True)
            plt.savefig(f'saliency_class_{target_label}_idx_{idx}.png')
            plt.close()
            
            print(f"已保存类别 {target_label}, 索引 {idx} 的显著性图")
            
    except ImportError:
        print("需要安装captum: pip install captum")


In [ ]:

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设置超参数
batch_size = 128
input_dim = 341
num_classes = 102

# 生成时间戳，用于保存文件
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f'./output/{timestamp}'
os.makedirs(output_dir, exist_ok=True)

# 初始化数据加载器 - 启用交叉验证
data_loader = BrainVoxelDataLoader(
    base_dir='./reorganized_data',  # 修改为实际路径
    cross_validation=True,  # 启用交叉验证
    cv_fold=30,  # 30折交叉验证
    standardize=True,  # 启用标准化
    output_dir=output_dir,  # 使用带时间戳的输出目录
    shuffle=True,  # 打乱训练集
    random_seed=42,  # 随机种子
    save_scaler=True  # 保存标准化器
)

# 执行30折交叉验证
cv_results = []

for fold in range(30):
    print(f"\n开始交叉验证第 {fold+1}/30 折")
    
    # 获取当前折的训练集和验证集
    train_dataset, val_dataset = data_loader.get_cv_fold(fold)
    
    # 获取测试集（所有折共用相同的测试集）
    test_dataset = data_loader.get_test_dataset()
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(val_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 为当前折创建目录
    fold_dir = os.path.join(output_dir, f'fold_{fold+1}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 训练模型
    model, history = train_model(
        train_loader, 
        val_loader, 
        input_dim=input_dim, 
        num_classes=num_classes, 
        device=device
    )
    
    # 评估模型
    test_loss, test_acc, y_pred, y_true = evaluate_model(model, test_loader, device=device)
    
    # 保存模型和结果
    torch.save(model.state_dict(), os.path.join(fold_dir, 'model.pth'))
    
    # 将当前折的结果添加到交叉验证结果中
    fold_result = {
        'fold': fold + 1,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'history': history
    }
    cv_results.append(fold_result)
    
    # 保存当前折的标准化器（带时间戳和折号）
    if data_loader.scaler is not None:
        scaler_path = os.path.join(fold_dir, f'scaler_{timestamp}_fold_{fold+1}.pkl')
        with open(scaler_path, 'wb') as f:
            pickle.dump(data_loader.scaler, f)
        print(f"标准化器已保存到: {scaler_path}")

# 保存所有交叉验证结果
with open(os.path.join(output_dir, f'cv_results_{timestamp}.pkl'), 'wb') as f:
    pickle.dump(cv_results, f)

# 计算并显示交叉验证平均结果
mean_test_loss = np.mean([result['test_loss'] for result in cv_results])
mean_test_acc = np.mean([result['test_acc'] for result in cv_results])
std_test_loss = np.std([result['test_loss'] for result in cv_results])
std_test_acc = np.std([result['test_acc'] for result in cv_results])

print(f"\n交叉验证平均测试损失: {mean_test_loss:.4f} ± {std_test_loss:.4f}")
print(f"交叉验证平均测试准确率: {mean_test_acc:.2f}% ± {std_test_acc:.2f}%")

# 绘制交叉验证结果
plt.figure(figsize=(10, 6))
plt.errorbar(
    range(1, 31), 
    [result['test_acc'] for result in cv_results], 
    yerr=std_test_acc, 
    fmt='o-', 
    capsize=5
)
plt.axhline(y=mean_test_acc, color='r', linestyle='--', label=f'Mean: {mean_test_acc:.2f}%')
plt.title('Cross-Validation Results: Test Accuracy')
plt.xlabel('Fold')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()
plt.savefig(os.path.join(output_dir, f'cv_results_{timestamp}.png'))

print(f"交叉验证结果已保存到 {output_dir}")
